# 04 — Deep-FCN Head Training (Focused JRI Reproduction)

This is a **reduced, standalone extraction** of the original `04_fusion_and_ablations.ipynb`. The JRI article compares *deep-only* classifiers, so handcrafted features, CLA and fusion ablations are intentionally omitted.

- Requires `00`, `02`, `03` (dataset manifest, splits, Swin checkpoints, and 768-D embeddings).
- For each fold, StandardScaler is fitted **only on its training embeddings**.
- FCN head architecture: `768→256→128→3`, same as the original `ffe_core.DeepOnlyNetFactory`.
- Original training procedure and early-stopping rule are retained. Existing valid checkpoints are reused unless `RETRAIN_HEADS=True`.
- Saves `deep_scaler_fold_XX.joblib`, `fold_XX/deep_fcn.pt`, and test predictions in the same paths expected by notebooks `08` and `09`.

This source notebook has *not* been re-executed after extraction. For published results, use the already-verified original checkpoints and notebook 08 baseline parity test.


In [ ]:
%pip -q install "scikit-learn==1.6.1" joblib

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, os

drive.mount('/content/drive')
PROJECT=Path(os.environ.get('FFE_PROJECT','/content/drive/MyDrive/tugas_akhir/FFE_Publication_2026'))
assert PROJECT.exists(), f'Folder proyek tidak ditemukan: {PROJECT}'
os.environ['FFE_ROOT']=str(PROJECT)
sys.path.insert(0,str(PROJECT/'02_notebooks'))
from ffe_core import ROOT, paths, seed_all, ensure_dataset_local, load_manifest, load_split
P=paths(); seed_all()
print('Project:',ROOT)

In [ ]:
import torch, numpy as np, pandas as pd, joblib, copy
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from ffe_core import DeepOnlyNetFactory, read_pack, save_prediction, metric_row, predict_logits

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS=80; PATIENCE=10; BATCH=64; LR=.001
RETRAIN_HEADS=False


## Model fitting (training only; validation-based early stopping)

In [ ]:
def fit_head(net,Xt,Ht,yt,Xv,Hv,yv,seed=42):
 seed_all(seed)
 net=net.to(DEVICE)
 cls=np.bincount(yt,minlength=3).astype(float)
 weights=len(yt)/np.maximum(3*cls,1.)
 criterion=torch.nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
 opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=1e-4)
 def get_loader(X,H,y,shuffle):
  tensors=[torch.as_tensor(X,dtype=torch.float32)]
  if H is not None:tensors.append(torch.as_tensor(H,dtype=torch.float32))
  tensors.append(torch.as_tensor(y,dtype=torch.int64))
  return DataLoader(TensorDataset(*tensors),batch_size=BATCH,shuffle=shuffle)
 tr=get_loader(Xt,Ht,yt,True);va=get_loader(Xv,Hv,yv,False)
 bestloss=float('inf');best=None;wait=0;hist=[]
 for epoch in range(EPOCHS):
  net.train()
  for batch in tr:
   *inputs,y=batch
   inputs=[x.to(DEVICE) for x in inputs];y=y.to(DEVICE)
   opt.zero_grad(set_to_none=True)
   logits=net(*inputs);loss=criterion(logits,y)
   loss.backward();opt.step()
  net.eval();total=0.
  with torch.no_grad():
   for batch in va:
    *inputs,y=batch;inputs=[x.to(DEVICE) for x in inputs];y=y.to(DEVICE)
    total+=criterion(net(*inputs),y).item()*len(y)
  total/=len(yv);hist.append({'epoch':epoch+1,'val_loss':total})
  if total<bestloss-1e-5:bestloss=total;best=copy.deepcopy(net.state_dict());wait=0
  else:
   wait+=1
   if wait>=PATIENCE:break
 net.load_state_dict(best);net.eval()
 return net,pd.DataFrame(hist)

## Train or reload each fold, then save FCN predictions

In [ ]:
for fold in range(1,6):
    print('Fold',fold)
    d={part:load_split(fold,part) for part in ('train','val','test')}
    pack={part:read_pack(P['features']/f'fold_{fold:02d}'/f'{part}_deep.npz',d[part]) for part in d}
    ds=StandardScaler().fit(pack['train']['X'])
    joblib.dump(ds,P['models']/f'deep_scaler_fold_{fold:02d}.joblib')
    Xd={part:ds.transform(pack[part]['X']).astype('float32') for part in d}
    model_dir=P['models']/f'fold_{fold:02d}'
    model_dir.mkdir(parents=True,exist_ok=True)
    ckpt=model_dir/'deep_fcn.pt'
    net=DeepOnlyNetFactory.create()
    if ckpt.exists() and not RETRAIN_HEADS:
        obj=torch.load(ckpt,map_location='cpu',weights_only=False)
        net.load_state_dict(obj['state_dict'])
        net=net.to(DEVICE).eval()
    else:
        net,hist=fit_head(net,Xd['train'],None,d['train'].label.to_numpy(),
                          Xd['val'],None,d['val'].label.to_numpy(),seed=42+fold)
        torch.save({'state_dict':net.state_dict(),'model':'deep_fcn','indices':None,'fold':fold},ckpt)
        hist.to_csv(P['logs']/f'deep_fcn_fold_{fold:02d}.csv',index=False)
    logits=predict_logits(net,Xd['test'],device=DEVICE)
    prob=torch.softmax(torch.from_numpy(logits),dim=1).numpy()
    pred=prob.argmax(1)
    save_prediction(P['metrics']/f'pred_deep_fcn_fold_{fold:02d}.csv',d['test'],pred,prob,'deep_fcn',fold)
    print(metric_row(d['test'].label,pred))
    del net
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
print('FCN models and predictions are ready for 08 and 09.')
